# Question Generation Pipeline

This notebook generates Vietnamese questions only from financial news articles using Gemini batch processing.

---

## Table of Contents

1. [Setup & Installation](#1-setup--installation)
2. [Configuration](#2-configuration)
   - Paths, model settings, schema
   - Q generation config and prompt
3. [Data Loading](#3-data-loading)
4. [Batch Processing Utilities](#4-batch-processing-utilities)
5. [Question Generation](#5-question-generation)

---

In [ ]:
# %%capture
# %pip install -q -U google-genai
# %pip install google-cloud-aiplatform

## 1. Setup & Installation

In [ ]:
from google import genai
import os
import pandas as pd
from google.genai import types
import json
from pydantic import RootModel, Field
from typing import List, Dict, Callable, Optional, Any
import numpy as np
import time

## 2. Configuration

Includes:
- **Paths**: Input/output and cache file paths
- **Model Settings**: API key, model name, batch size
- **Schema**: `QuestionOutput` for JSON validation
- **Generation Config**: `Q_GEN_CONFIG`
- **Prompt**: `Q_INSTRUCTION`

In [ ]:
import os
# ==================== CONFIGURATION ====================
# Paths
INPUT_PATH = "../data/processed/cleaned_classified_segmented_data.parquet"
BATCH_Q_PATH = "../data/cache/batch_generate_questions_{}.jsonl"
OUTPUT_Q_PATH = "../data/processed/Q_data.parquet"

# Cache paths (incremental saving/loading)
CACHE_SAVE_FOLDER = "../data/cache/"
CACHE_LOAD_FOLDER = "../data/cache/"
os.makedirs(CACHE_SAVE_FOLDER, exist_ok=True)

# Resume configuration
RESUME_FROM_LAST_CHECKPOINT = True

# Model settings
MODEL_NAME = "gemini-2.5-flash"
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# Batch processing
BATCH_CHUNK_SIZE = 1000
RANDOM_STATE = 42

# Q Output Class
class QuestionOutput(RootModel):
    root: List[str] = Field(
        ...,
        min_length=1,
        max_length=2,
        description="List 1-2 chuoi. BAT BUOC format: '[ID]-[Noi dung cau hoi]' (Vi du: '1-Ty gia la bao nhieu?')"
    )

# Generation config
Q_GEN_CONFIG = {
    "max_output_tokens": 200,
    "temperature": 0.7,
    "top_p": 0.9,
    "response_mime_type": "application/json",
    "response_json_schema": QuestionOutput.model_json_schema(),
    "thinking_config": {"thinking_budget": 0}
}

# ==================== PROMPTS ====================
Q_INSTRUCTION = """You are an expert at creating questions for Vietnamese financial QA systems.

## CONTEXT INFORMATION
**Article Title:** {title}
**Publication Date:** {timestamp}

**Content:**
{text}

## TASK
Create 1-2 high-quality questions based ENTIRELY on the content above.

## MANDATORY RULES
1. **STRICTLY FORBIDDEN to use external knowledge** - Only use information in the context
2. Questions must have clear answers within the content
3. Do not ask about vague or uncertain information
4. Do not create questions requiring complex calculations
5. Prioritize important, core information from the article
6. **ALL questions MUST be written in Vietnamese**

## QUESTION TYPES

Question types are categorized below. Choose the most appropriate types based on the context content:

### 1. Factoid Extraction
- Ask about: numbers, company names, ratios, values, people, SPECIFIC times
- Example: "Loi nhuan rong cua VPBank trong Q3 2024 la bao nhieu?"

### 2. Summary & Interpretation
- Ask about: causes, impacts, trends, meanings mentioned
- Example: "Nhung yeu to nao duoc de cap anh huong den ket qua kinh doanh?"

### 3. Comparison
- Compare numbers, periods, entities WITHIN the same text
- Example: "Doanh thu Q3 tang hay giam so voi Q2?"

### 4. Verification/Confirmation
- Yes/No questions or information verification
- Example: "Cong ty co chi tra co tuc trong ky nay khong?"

## SELECTION STRATEGY
- If text has many numbers -> prioritize Factoid + Comparison
- If text analyzes trends -> prioritize Interpretation + Factoid
- If text is short, simple -> prioritize Factoid + Verification

## OUTPUT FORMAT (MANDATORY)
Return JSON array with format: ["[ID]-[Question content in Vietnamese]"]

## IMPORTANT NOTES
- Questions must be natural and clear like real people asking
- Select the 2 most relevant question types for this specific content
- Avoid too easy questions (answerable with 1 word) or too hard (requiring multi-step reasoning)
- If context lacks sufficient information for quality questions, ONLY create 1 best question
- **Remember: Return ALL questions in Vietnamese**

Now create the questions:"""

# ==================== CLIENT ====================
client = genai.Client(api_key=GEMINI_API_KEY)

## 3. Data Loading

Load the input data and sample for processing.

In [ ]:
df = pd.read_parquet(INPUT_PATH, engine="pyarrow").sample(n=100, random_state=RANDOM_STATE)

In [ ]:
inputs = []
print("Processing initial inputs...")
for index, row in df.iterrows():
    chunks = row['chunks']
    title = row['title']
    url = row['url']
    timestamp = row['time']
    
    num_chunks = len(chunks)
    
    if num_chunks == 2:
        # 1-2 chunks: use all
        selected_indices = list(range(num_chunks))
    else:
        # 3+ chunks: randomly select 2 chunks
        selected_indices = list(np.random.choice(num_chunks, size=2, replace=False))
    
    # Add selected chunks to inputs
    for chunk_idx in selected_indices:
        inputs.append({
            'index': index,
            'chunk_idx': chunk_idx,
            'url': url,
            'title': title,
            'time': timestamp,
            'text': chunks[chunk_idx]
        })

print(f"Total inputs to process: {len(inputs)}")

## 4. Batch Processing Utilities

Reusable functions for batch file creation, job submission, and result parsing.

In [ ]:
# ==================== BATCH PROCESSING UTILITIES ====================

def upload_and_submit_batch(filename: str) -> Any:
    """Upload a file and create a batch job."""
    print(f"[INFO] Uploading {filename}...")
    uploaded = client.files.upload(
        file=filename,
        config=types.UploadFileConfig(
            display_name="generation-requests",
            mime_type="text/plain"
        ),
    )
    print(f"[INFO] Creating batch job for {uploaded.name}...")
    batch_job = client.batches.create(
        model=MODEL_NAME,
        src=uploaded.name,
        config={"display_name": "generation-run"},
    )
    print(f"[INFO] Job created: {batch_job.name}")
    return batch_job


def wait_for_job_completion(job_name: str, poll_interval: int = 30) -> Any:
    """Poll for batch job completion with progress updates."""
    print(f"[INFO] Waiting for job {job_name}...")
    start_time = time.time()
    last_log_time = start_time
    
    while True:
        job = client.batches.get(name=job_name)
        state = job.state.name
        
        # Log progress every 60 seconds
        current_time = time.time()
        if current_time - last_log_time >= 60:
            elapsed = (current_time - start_time) / 60
            print(f"[INFO] Job {job_name} - State: {state} - Elapsed: {elapsed:.1f} min")
            last_log_time = current_time
        
        if state not in ("JOB_STATE_RUNNING", "JOB_STATE_PENDING"):
            break
        time.sleep(poll_interval)
    
    elapsed = (time.time() - start_time) / 60
    if state == "JOB_STATE_SUCCEEDED":
        print(f"[SUCCESS] Job {job_name} completed in {elapsed:.1f} min")
    else:
        print(f"[ERROR] Job {job_name} ended with state: {state} after {elapsed:.1f} min")
    return job


def create_batch_files(
    inputs: List[Dict],
    batch_path_template: str,
    prompt_formatter: Callable[[Dict], str],
    gen_config: Dict,
    chunk_size: int = BATCH_CHUNK_SIZE
) -> List[str]:
    """Create JSONL batch files from inputs."""
    filenames = []
    file_idx = 1
    
    for i in range(0, len(inputs), chunk_size):
        batch = inputs[i:i + chunk_size]
        filename = batch_path_template.format(file_idx)
        file_idx += 1
        
        with open(filename, "w", encoding="utf-8") as f:
            for batch_idx, item in enumerate(batch):
                global_idx = i + batch_idx
                record = {
                    "key": str(global_idx),
                    "request": {
                        "contents": [{"parts": [{"text": prompt_formatter(item)}]}],
                        "generation_config": gen_config,
                    },
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
        
        filenames.append(filename)
    
    print(f"[INFO] Created {len(filenames)} batch files ({len(inputs)} total items)")
    return filenames


def parse_batch_results(
    records_list: List[List[Dict]],
    parser: Optional[Callable[[str], Any]] = None
) -> tuple[Dict, List[Dict]]:
    """Parse batch results with optional custom parser."""
    results = {}
    errors = []
    
    for records in records_list:
        for rec in records:
            try:
                key = rec.get("key", "")
                result_text = rec["response"]["candidates"][0]["content"]["parts"][0]["text"]
                if parser:
                    results[key] = parser(result_text)
                else:
                    results[key] = result_text.strip()
            except Exception as e:
                errors.append({"record": rec, "error": str(e)})
    
    print(f"[INFO] Successfully parsed: {len(results)} results")
    print(f"[INFO] Errors: {len(errors)}")
    return results, errors


def process_single_batch(
    filename: str,
    job_name: str,
    parser: Optional[Callable[[str], Any]] = None
) -> tuple[Dict, List[Dict]]:
    """Download, parse, and return results from a single completed batch job."""
    try:
        print(f"[INFO] Downloading results for {filename} (job: {job_name})...")
        job = client.batches.get(name=job_name)
        result_file = job.dest.file_name
        raw = client.files.download(file=result_file).decode("utf-8")
        records = [json.loads(line) for line in raw.splitlines() if line.strip()]
        
        print(f"[INFO] Downloaded {len(records)} records from {filename}")
        results, errors = parse_batch_results([records], parser=parser)
        return results, errors
    except Exception as e:
        print(f"[ERROR] Failed to download/parse {filename}: {e}")
        return {}, []


def save_incremental_results(batch_idx: int, results: Dict, prefix: str = "qgen") -> str:
    """Save batch results incrementally to cache folder (results only)."""
    cache_path = os.path.join(CACHE_SAVE_FOLDER, f"{prefix}_results_batch_{batch_idx}.json")
    with open(cache_path, "w", encoding="utf-8") as cache_file:
        json.dump(results, cache_file, ensure_ascii=False, indent=4)
    print(f"[INFO] Saved batch {batch_idx} to: {cache_path}")
    return cache_path


def load_incremental_results(
    start_batch: int = 1,
    end_batch: Optional[int] = None,
    prefix: str = "qgen"
) -> tuple[Dict, int]:
    """Load previously saved results from cache folder."""
    merged_results = {}
    last_batch_idx = start_batch - 1
    
    batch_idx = start_batch
    while True:
        if end_batch is not None and batch_idx > end_batch:
            break
        
        cache_path = os.path.join(CACHE_LOAD_FOLDER, f"{prefix}_results_batch_{batch_idx}.json")
        if not os.path.exists(cache_path):
            if batch_idx == start_batch:
                print(f"[WARNING] No cache file found at batch {batch_idx}")
            break
        
        try:
            with open(cache_path, "r", encoding="utf-8") as cache_file:
                batch_results = json.load(cache_file)
            merged_results.update(batch_results)
            last_batch_idx = batch_idx
            
            print(f"[INFO] Loaded batch {batch_idx}: {len(batch_results)} results")
            batch_idx += 1
        except Exception as e:
            print(f"[ERROR] Failed to load batch {batch_idx}: {e}")
            break
    
    if last_batch_idx >= start_batch:
        print(f"\n[SUCCESS] Loaded batches {start_batch}-{last_batch_idx}")
        print(f"[INFO] Total merged results: {len(merged_results)}")
    else:
        print("[INFO] No batches loaded")
    
    return merged_results, last_batch_idx

## 5. Question Generation

In [ ]:
# Define prompt formatter for question generation
def format_question_prompt(item: Dict) -> str:
    return Q_INSTRUCTION.format(
        title=item["title"],
        timestamp=item["time"],
        text=item["text"]
    )

def parse_questions(result_text: str) -> List[str]:
    questions_obj = QuestionOutput.model_validate_json(result_text)
    return questions_obj.root

# Create batch files
q_filenames = create_batch_files(
    inputs=inputs,
    batch_path_template=BATCH_Q_PATH,
    prompt_formatter=format_question_prompt,
    gen_config=Q_GEN_CONFIG
)

# ==================== INCREMENTAL BATCH PROCESSING ====================
if RESUME_FROM_LAST_CHECKPOINT:
    print("\n[INFO] Resuming from last checkpoint...")
    q_generation_results, last_batch = load_incremental_results(prefix="qgen")
    print(f"\n[INFO] Resume point: Last completed batch = {last_batch}")
else:
    q_generation_results = {}
    last_batch = 0

start_batch = last_batch + 1
q_errors = []

print(f"[INFO] Processing {len(q_filenames)} files...")
print(f"[INFO] Will start from batch {start_batch}")

for batch_idx, filename in enumerate(q_filenames, start=1):
    print(f"\n{'='*60}")
    print(f"[Q-GEN] Processing batch {batch_idx}/{len(q_filenames)}: {filename}")
    print(f"{'='*60}")
    
    if batch_idx <= last_batch:
        print(f"[INFO] Skipping batch {batch_idx} as it is before resume point")
        continue
    
    cache_path = os.path.join(CACHE_SAVE_FOLDER, f"qgen_results_batch_{batch_idx}.json")
    if os.path.exists(cache_path):
        print(f"[INFO] Batch {batch_idx} already processed. Loading from cache...")
        try:
            with open(cache_path, "r", encoding="utf-8") as f:
                results = json.load(f)
            q_generation_results.update(results)
            print(f"[Q-GEN] Batch {batch_idx}/{len(q_filenames)} loaded from cache")
            print(f"[Q-GEN] Total results so far: {len(q_generation_results)}")
            continue
        except Exception as e:
            print(f"[WARNING] Failed to load cache for batch {batch_idx}: {e}")
            print(f"[INFO] Reprocessing batch {batch_idx}...")
    
    # Step 1: Submit batch job
    job = upload_and_submit_batch(filename)
    
    # Step 2: Wait for completion
    completed_job = wait_for_job_completion(job.name)
    if completed_job.state.name != "JOB_STATE_SUCCEEDED":
        print(f"[ERROR] Batch {batch_idx} failed, skipping download")
        continue
    
    # Step 3: Download and parse results
    results, errors = process_single_batch(
        filename=filename,
        job_name=job.name,
        parser=parse_questions
    )
    
    # Step 4: Merge results
    q_generation_results.update(results)
    q_errors.extend(errors)
    
    # Step 5: Save incrementally
    save_incremental_results(batch_idx, results, prefix="qgen")
    
    print(f"[Q-GEN] Batch {batch_idx}/{len(q_filenames)} completed")
    print(f"[Q-GEN] Total results so far: {len(q_generation_results)}")

print(f"\n{'='*60}")
print(f"[SUCCESS] All {len(q_filenames)} batch files processed!")
print(f"[INFO] Total results: {len(q_generation_results)}")
print(f"[INFO] Total errors: {len(q_errors)}")
print(f"{'='*60}\n")

# ==================== SAVE INTERMEDIATE RESULTS ====================
intermediate_data = []
for idx, item in enumerate(inputs):
    key = str(idx)
    questions = q_generation_results.get(key, [])
    intermediate_data.append({
        "input_idx": idx,
        "article_idx": item["index"],
        "chunk_idx": item["chunk_idx"],
        "url": item["url"],
        "title": item["title"],
        "time": item["time"],
        "context": item["text"],
        "questions": questions
    })

df_questions = pd.DataFrame(intermediate_data)
df_questions.to_parquet(OUTPUT_Q_PATH, engine="pyarrow", index=False)
print(f"[INFO] Saved questions to: {OUTPUT_Q_PATH}")
print(f"[INFO] Total chunks with questions: {len(df_questions)}")
print(f"[INFO] Total questions generated: {sum(len(q) for q in df_questions['questions'])}")